In [3]:
!python -m pip install openai pandas python-dotenv

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.6 MB/s  0:00:00
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 36.4 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached typing_inspection-0.4.2-py3-none-any.whl (14 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [openai]2m8/9 [openai]c]


In [4]:
import os
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

In [5]:
PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
API_OUTPUTS_DIR = PROJECT_ROOT / "data" / "api_outputs"

API_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROCESSED_DIR exists?", PROCESSED_DIR.exists())
print("API_OUTPUTS_DIR exists?", API_OUTPUTS_DIR.exists())

PROCESSED_DIR exists? True
API_OUTPUTS_DIR exists? True


In [6]:
all_df = pd.read_csv(PROCESSED_DIR / "texts_with_clusters.csv")
print(all_df.shape)
all_df.head()

(1793, 12)


,dataset,source,page_title,text_id,raw_text,cleaned_text,word_count,dataset_label,cluster,pca_x,pca_y,cluster_name
0,landforms,wikipedia,Desert,Desert_0,A desert is a landscape where little precipita...,A desert is a landscape where little precipita...,91,landforms,5,0.371352,0.011644,atmospheric_abstract
1,landforms,wikipedia,Desert,Desert_1,Deserts are formed by weathering processes as ...,Deserts are formed by weathering processes as ...,218,landforms,5,0.358664,0.273669,atmospheric_abstract
2,landforms,wikipedia,Desert,Desert_2,Plants and animals living in the desert need s...,Plants and animals living in the desert need s...,147,landforms,5,0.173692,-0.226674,atmospheric_abstract
3,landforms,wikipedia,Desert,Desert_3,People have struggled to live in deserts and t...,People have struggled to live in deserts and t...,151,landforms,5,0.042568,-0.075830,atmospheric_abstract
4,landforms,wikipedia,Desert,Desert_4,English desert and its Romance cognates (inclu...,English desert and its Romance cognates (inclu...,131,landforms,5,0.096850,-0.023956,atmospheric_abstract


In [7]:
cluster_examples = {}

for c in sorted(all_df["cluster"].unique()):
    sub = all_df[all_df["cluster"] == c].copy()

    cluster_name = sub["cluster_name"].iloc[0] if "cluster_name" in sub.columns else f"cluster_{c}"

    examples = sub["cleaned_text"].dropna().astype(str).head(8).tolist()

    cluster_examples[str(c)] = {
        "cluster_id": int(c),
        "cluster_name": cluster_name,
        "dataset_distribution": sub["dataset_label"].value_counts().to_dict(),
        "examples": examples
    }

cluster_examples

{'0': {'cluster_id': 0,
  'cluster_name': 'dry_exposed',
  'dataset_distribution': {'landforms': 146, 'climate': 11, 'gutenberg': 3},
  'examples': ['Cave-inhabiting animals are often categorized as troglobites (cave-limited species), troglophiles (species that can live their entire lives in caves, but also occur in other environments), trogloxenes (species that use caves, but cannot complete their life cycle fully in caves) and accidentals (animals not in one of the previous categories). Some authors use separate terminology for aquatic forms (for example, stygobites, stygophiles, and stygoxenes).',
   'Of these animals, the troglobites are perhaps the most unusual organisms. Troglobitic species often show a number of characteristics, termed troglomorphic, associated with their adaptation to subterranean life. These characteristics may include a loss of pigment (often resulting in a pale or white coloration), a loss of eyes (or at least of optical functionality), an elongation of appe

In [8]:
with open(API_OUTPUTS_DIR / "cluster_examples.json", "w", encoding="utf-8") as f:
    json.dump(cluster_examples, f, ensure_ascii=False, indent=2)

print("Saved:", API_OUTPUTS_DIR / "cluster_examples.json")

Saved: /Users/apple/Desktop/UCL/skills/python3/data/api_outputs/cluster_examples.json


In [ ]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded?", OPENAI_API_KEY is not None)

In [9]:
!python -m pip install python-dotenv

In [10]:
import os
from dotenv import load_dotenv
from pathlib import Path

In [11]:
PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
load_dotenv(PROJECT_ROOT / ".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded?", OPENAI_API_KEY is not None)

API key loaded? True


In [12]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
print("client created")

client created


In [13]:
import pandas as pd
from pathlib import Path
import json

PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
API_OUTPUTS_DIR = PROJECT_ROOT / "data" / "api_outputs"

API_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

all_df = pd.read_csv(PROCESSED_DIR / "texts_with_clusters.csv")
print(all_df.shape)
print(all_df.columns.tolist())
all_df.head()

(1793, 12)
['dataset', 'source', 'page_title', 'text_id', 'raw_text', 'cleaned_text', 'word_count', 'dataset_label', 'cluster', 'pca_x', 'pca_y', 'cluster_name']


,dataset,source,page_title,text_id,raw_text,cleaned_text,word_count,dataset_label,cluster,pca_x,pca_y,cluster_name
0,landforms,wikipedia,Desert,Desert_0,A desert is a landscape where little precipita...,A desert is a landscape where little precipita...,91,landforms,5,0.371352,0.011644,atmospheric_abstract
1,landforms,wikipedia,Desert,Desert_1,Deserts are formed by weathering processes as ...,Deserts are formed by weathering processes as ...,218,landforms,5,0.358664,0.273669,atmospheric_abstract
2,landforms,wikipedia,Desert,Desert_2,Plants and animals living in the desert need s...,Plants and animals living in the desert need s...,147,landforms,5,0.173692,-0.226674,atmospheric_abstract
3,landforms,wikipedia,Desert,Desert_3,People have struggled to live in deserts and t...,People have struggled to live in deserts and t...,151,landforms,5,0.042568,-0.075830,atmospheric_abstract
4,landforms,wikipedia,Desert,Desert_4,English desert and its Romance cognates (inclu...,English desert and its Romance cognates (inclu...,131,landforms,5,0.096850,-0.023956,atmospheric_abstract


In [14]:
cluster_examples = {}

for c in sorted(all_df["cluster"].unique()):
    sub = all_df[all_df["cluster"] == c].copy()

    cluster_name = sub["cluster_name"].iloc[0] if "cluster_name" in sub.columns else f"cluster_{c}"

    examples = sub["cleaned_text"].dropna().astype(str).head(8).tolist()

    if "dataset_label" in sub.columns:
        dataset_distribution = sub["dataset_label"].value_counts().to_dict()
    else:
        dataset_distribution = {}

    cluster_examples[str(c)] = {
        "cluster_id": int(c),
        "cluster_name": cluster_name,
        "dataset_distribution": dataset_distribution,
        "examples": examples
    }

cluster_examples

{'0': {'cluster_id': 0,
  'cluster_name': 'dry_exposed',
  'dataset_distribution': {'landforms': 146, 'climate': 11, 'gutenberg': 3},
  'examples': ['Cave-inhabiting animals are often categorized as troglobites (cave-limited species), troglophiles (species that can live their entire lives in caves, but also occur in other environments), trogloxenes (species that use caves, but cannot complete their life cycle fully in caves) and accidentals (animals not in one of the previous categories). Some authors use separate terminology for aquatic forms (for example, stygobites, stygophiles, and stygoxenes).',
   'Of these animals, the troglobites are perhaps the most unusual organisms. Troglobitic species often show a number of characteristics, termed troglomorphic, associated with their adaptation to subterranean life. These characteristics may include a loss of pigment (often resulting in a pale or white coloration), a loss of eyes (or at least of optical functionality), an elongation of appe

In [15]:
with open(API_OUTPUTS_DIR / "cluster_examples.json", "w", encoding="utf-8") as f:
    json.dump(cluster_examples, f, ensure_ascii=False, indent=2)

print("Saved:", API_OUTPUTS_DIR / "cluster_examples.json")

Saved: /Users/apple/Desktop/UCL/skills/python3/data/api_outputs/cluster_examples.json


In [16]:
design_schema = {
    "name": "cluster_design_translation",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "cluster_id": {"type": "integer"},
            "cluster_name": {"type": "string"},
            "semantic_summary": {"type": "string"},
            "openness": {"type": "integer", "minimum": 1, "maximum": 10},
            "verticality": {"type": "integer", "minimum": 1, "maximum": 10},
            "roughness": {"type": "integer", "minimum": 1, "maximum": 10},
            "density": {"type": "integer", "minimum": 1, "maximum": 10},
            "motion_intensity": {"type": "integer", "minimum": 1, "maximum": 10},
            "fragmentation": {"type": "integer", "minimum": 1, "maximum": 10},
            "surface_continuity": {"type": "integer", "minimum": 1, "maximum": 10},
            "material_impression": {"type": "string"},
            "atmosphere_keywords": {
                "type": "array",
                "items": {"type": "string"}
            },
            "tonal_direction": {"type": "string"},
            "image_prompt": {"type": "string"},
            "td_texture_prompt": {"type": "string"},
            "blender_fragment_prompt": {"type": "string"}
        },
        "required": [
            "cluster_id",
            "cluster_name",
            "semantic_summary",
            "openness",
            "verticality",
            "roughness",
            "density",
            "motion_intensity",
            "fragmentation",
            "surface_continuity",
            "material_impression",
            "atmosphere_keywords",
            "tonal_direction",
            "image_prompt",
            "td_texture_prompt",
            "blender_fragment_prompt"
        ],
        "additionalProperties": False
    }
}

In [17]:
def build_cluster_prompt(cluster_payload: dict) -> str:
    cluster_id = cluster_payload["cluster_id"]
    cluster_name = cluster_payload["cluster_name"]
    examples = cluster_payload["examples"]
    dataset_distribution = cluster_payload["dataset_distribution"]

    example_block = "\n\n".join(
        [f"Example {i+1}: {txt}" for i, txt in enumerate(examples)]
    )

    prompt = f"""
You are helping translate clustered environmental language into structured design parameters for a data-driven 3D design workflow.

Cluster ID: {cluster_id}
Cluster Name: {cluster_name}
Dataset Distribution: {dataset_distribution}

Below are representative text examples from this cluster:

{example_block}

Your task:
1. Identify the dominant environmental and spatial qualities in these texts.
2. Summarise the semantic character of the cluster.
3. Translate the cluster into numerical design parameters from 1 to 10.
4. Generate one concise abstract image prompt for grayscale / terrain-like visual generation.
5. Generate one prompt specifically for TouchDesigner texture generation.
6. Generate one prompt specifically for Blender fragment generation.

Important constraints:
- Focus on spatial and material qualities, not narrative plot.
- Keep the output suitable for abstract architectural fragment design.
- Avoid mentioning people, characters, or literary storytelling.
- The image prompt should favour abstract, monochrome, environmental texture.
- The TouchDesigner prompt should describe texture logic, contrast, erosion, flow, grain, or density.
- The Blender prompt should describe fragment form, height variation, edge character, density, and continuity.
"""
    return prompt.strip()

In [24]:
import json

def translate_cluster_with_openai(cluster_payload: dict, model_name="gpt-4.1-mini"):
    prompt = build_cluster_prompt(cluster_payload)

    response = client.responses.create(
        model=model_name,
        input=[
            {
                "role": "system",
                "content": [
                    {
                        "type": "input_text",
                        "text": "Return only valid JSON that matches the provided schema."
                    }
                ]
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_text",
                        "text": prompt
                    }
                ]
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": design_schema["name"],
                "schema": design_schema["schema"],
                "strict": True
            }
        }
    )

    print(response.output_text)
    return json.loads(response.output_text)

In [19]:
test_cluster = cluster_examples["0"]
print(build_cluster_prompt(test_cluster)[:1500])

You are helping translate clustered environmental language into structured design parameters for a data-driven 3D design workflow.

Cluster ID: 0
Cluster Name: dry_exposed
Dataset Distribution: {'landforms': 146, 'climate': 11, 'gutenberg': 3}

Below are representative text examples from this cluster:

Example 1: Cave-inhabiting animals are often categorized as troglobites (cave-limited species), troglophiles (species that can live their entire lives in caves, but also occur in other environments), trogloxenes (species that use caves, but cannot complete their life cycle fully in caves) and accidentals (animals not in one of the previous categories). Some authors use separate terminology for aquatic forms (for example, stygobites, stygophiles, and stygoxenes).

Example 2: Of these animals, the troglobites are perhaps the most unusual organisms. Troglobitic species often show a number of characteristics, termed troglomorphic, associated with their adaptation to subterranean life. These 

In [21]:
import os
from dotenv import load_dotenv
from pathlib import Path

PROJECT_ROOT = Path("/Users/apple/Desktop/UCL/skills/python3")
load_dotenv(PROJECT_ROOT / ".env", override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("Loaded?", OPENAI_API_KEY is not None)
print("Prefix:", OPENAI_API_KEY[:12] if OPENAI_API_KEY else None)
print("Suffix:", OPENAI_API_KEY[-8:] if OPENAI_API_KEY else None)
print("Length:", len(OPENAI_API_KEY) if OPENAI_API_KEY else None)

Loaded? True
Prefix: sk-proj--VsW
Suffix: QGfASpIA
Length: 164


In [22]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

response = client.responses.create(
    model="gpt-4.1-mini",
    input="Reply with exactly: API test successful."
)

print(response.output_text)

API test successful.


In [25]:
test_cluster = cluster_examples["0"]
print(build_cluster_prompt(test_cluster)[:1200])

You are helping translate clustered environmental language into structured design parameters for a data-driven 3D design workflow.

Cluster ID: 0
Cluster Name: dry_exposed
Dataset Distribution: {'landforms': 146, 'climate': 11, 'gutenberg': 3}

Below are representative text examples from this cluster:

Example 1: Cave-inhabiting animals are often categorized as troglobites (cave-limited species), troglophiles (species that can live their entire lives in caves, but also occur in other environments), trogloxenes (species that use caves, but cannot complete their life cycle fully in caves) and accidentals (animals not in one of the previous categories). Some authors use separate terminology for aquatic forms (for example, stygobites, stygophiles, and stygoxenes).

Example 2: Of these animals, the troglobites are perhaps the most unusual organisms. Troglobitic species often show a number of characteristics, termed troglomorphic, associated with their adaptation to subterranean life. These 

In [26]:
test_result = translate_cluster_with_openai(test_cluster)
test_result

{"cluster_id":0,"cluster_name":"dry_exposed","semantic_summary":"This cluster represents environments characterized by exposed, fragile geological and biological formations, such as caves and coral reefs, emphasizing delicate structural textures, mineral and biological remnants, and dynamic surface continuity shaped by water erosion and sedimentation processes. The spatial qualities suggest intricate open cavities and complex surfaces with varying verticality but overall low density, highlighting fragile, rough, and eroded textures typical of dry-exposed subterranean and marine ecosystems.","openness":7,"verticality":6,"roughness":8,"density":3,"motion_intensity":4,"fragmentation":7,"surface_continuity":5,"material_impression":"mineral crusts, calcium carbonate exoskeletons, eroded rock, biogenic structures","atmosphere_keywords":["fragile","eroded","textured","delicate","subterranean","marine","exposed","complex"],"tonal_direction":"mid-high contrast grayscale emphasizing texture dept

{'cluster_id': 0,
 'cluster_name': 'dry_exposed',
 'semantic_summary': 'This cluster represents environments characterized by exposed, fragile geological and biological formations, such as caves and coral reefs, emphasizing delicate structural textures, mineral and biological remnants, and dynamic surface continuity shaped by water erosion and sedimentation processes. The spatial qualities suggest intricate open cavities and complex surfaces with varying verticality but overall low density, highlighting fragile, rough, and eroded textures typical of dry-exposed subterranean and marine ecosystems.',
 'openness': 7,
 'verticality': 6,
 'roughness': 8,
 'density': 3,
 'motion_intensity': 4,
 'fragmentation': 7,
 'surface_continuity': 5,
 'material_impression': 'mineral crusts, calcium carbonate exoskeletons, eroded rock, biogenic structures',
 'atmosphere_keywords': ['fragile',
  'eroded',
  'textured',
  'delicate',
  'subterranean',
  'marine',
  'exposed',
  'complex'],
 'tonal_directi

In [27]:
cluster_design_outputs = {}

for cluster_key, cluster_payload in cluster_examples.items():
    print(f"Processing cluster {cluster_key} ...")
    try:
        result = translate_cluster_with_openai(cluster_payload)
        cluster_design_outputs[cluster_key] = result
        print("Done:", cluster_payload["cluster_name"])
    except Exception as e:
        print("Error on cluster", cluster_key, e)

Processing cluster 0 ...
{"cluster_id":0,"cluster_name":"dry_exposed","semantic_summary":"This cluster characterizes environments that are predominantly dry or exposed surfaces featuring rugged terrain and complex microhabitats, ranging from cave interiors to coral reef structures. Dominant qualities include porous, coarse textures with notable surface fragmentation and varied verticality due to geological and biological formations. The atmosphere suggests isolation, fragility, and exposure to natural erosive processes.","openness":4,"verticality":7,"roughness":8,"density":5,"motion_intensity":2,"fragmentation":7,"surface_continuity":4,"material_impression":"porous carbonate rock and dry cavernous substrate","atmosphere_keywords":["fragile","isolated","exposed","textured","weathered"],"tonal_direction":"muted monochrome with high texture contrast","image_prompt":"abstract grayscale terrain showing rugged, porous surfaces with deep fissures and fragmented, layered structures reminiscent

In [28]:
with open(API_OUTPUTS_DIR / "cluster_design_parameters.json", "w", encoding="utf-8") as f:
    json.dump(cluster_design_outputs, f, ensure_ascii=False, indent=2)

print("Saved:", API_OUTPUTS_DIR / "cluster_design_parameters.json")

Saved: /Users/apple/Desktop/UCL/skills/python3/data/api_outputs/cluster_design_parameters.json


In [29]:
rows = []

for k, v in cluster_design_outputs.items():
    row = v.copy()
    row["atmosphere_keywords"] = ", ".join(v.get("atmosphere_keywords", []))
    rows.append(row)

cluster_params_df = pd.DataFrame(rows)
cluster_params_df

,cluster_id,cluster_name,semantic_summary,openness,verticality,roughness,density,motion_intensity,fragmentation,surface_continuity,material_impression,atmosphere_keywords,tonal_direction,image_prompt,td_texture_prompt,blender_fragment_prompt
0,0,dry_exposed,This cluster characterizes environments that a...,4,7,8,5,2,7,4,porous carbonate rock and dry cavernous substrate,"fragile, isolated, exposed, textured, weathered",muted monochrome with high texture contrast,"abstract grayscale terrain showing rugged, por...",high contrast erosion pattern with layered gra...,dense cluster of varied height fragments with ...
1,1,frozen_quiet,"An expansive, stark environment characterized ...",9,2,3,2,1,3,7,"dry, granular, powdery sand and stone surfaces...","dry, quiet, bleak, isolated, expansive, stilln...",monochrome grayscale with soft contrasts and s...,vast monochrome desert terrain with gentle und...,generate sparse granular texture with low cont...,create low-height uneven terrain fragments wit...
2,2,lush_dense,The cluster characterizes extreme arid desert ...,9,3,9,2,4,8,3,"predominantly mineral, salt crusts, fragmented...","arid, desolate, barren, dry, harsh, wind-swept...",high contrast grayscale with sharp edges and s...,abstract monochrome terrain of cracked and fra...,generate high contrast erosion patterns with s...,create dense fields of irregular flat fragment...
3,3,flowing_unstable,"Deep, narrow, and unstable landforms shaped by...",2,9,8,4,7,8,3,"rocky, layered, eroded stone with varying hard...","deep, narrow, steep, rugged, eroded, dynamic, ...",high contrast grayscale with stark light and s...,"abstract monochrome terrain of a deep, narrow ...",high contrast procedural erosion texture simul...,fragmented vertical rock forms with high heigh...
4,4,rocky_vertical,"The cluster centers on harsh, rocky, and verti...",3,8,9,3,2,8,2,"rocky, granular, weathered stone with occasion...","barren, rugged, sparse vegetation, eroded, tex...",high contrast grayscale with strong shadows an...,Abstract terrain of rugged vertical rock faces...,Generate a high contrast procedural texture of...,Create fragmented vertical rock formations wit...
5,5,atmospheric_abstract,This cluster captures the spatial and material...,9,3,7,2,4,8,3,"dry, eroded stone, sand, cracked earth, salt c...","arid, dusty, eroded, fragmented, bleached, sta...",high contrast grayscale with emphasis on textu...,Abstract grayscale aerial terrain depicting va...,High-contrast texture featuring granular sand ...,Generate low to moderate height fragmented roc...


In [30]:
cluster_params_df.to_csv(API_OUTPUTS_DIR / "cluster_parameters_flat.csv", index=False)
print("Saved:", API_OUTPUTS_DIR / "cluster_parameters_flat.csv")

Saved: /Users/apple/Desktop/UCL/skills/python3/data/api_outputs/cluster_parameters_flat.csv


In [31]:
cluster_prompts_df = cluster_params_df[[
    "cluster_id",
    "cluster_name",
    "image_prompt",
    "td_texture_prompt",
    "blender_fragment_prompt"
]].copy()

cluster_prompts_df.to_csv(API_OUTPUTS_DIR / "cluster_prompts.csv", index=False)
print("Saved:", API_OUTPUTS_DIR / "cluster_prompts.csv")
cluster_prompts_df

Saved: /Users/apple/Desktop/UCL/skills/python3/data/api_outputs/cluster_prompts.csv


,cluster_id,cluster_name,image_prompt,td_texture_prompt,blender_fragment_prompt
0,0,dry_exposed,"abstract grayscale terrain showing rugged, por...",high contrast erosion pattern with layered gra...,dense cluster of varied height fragments with ...
1,1,frozen_quiet,vast monochrome desert terrain with gentle und...,generate sparse granular texture with low cont...,create low-height uneven terrain fragments wit...
2,2,lush_dense,abstract monochrome terrain of cracked and fra...,generate high contrast erosion patterns with s...,create dense fields of irregular flat fragment...
3,3,flowing_unstable,"abstract monochrome terrain of a deep, narrow ...",high contrast procedural erosion texture simul...,fragmented vertical rock forms with high heigh...
4,4,rocky_vertical,Abstract terrain of rugged vertical rock faces...,Generate a high contrast procedural texture of...,Create fragmented vertical rock formations wit...
5,5,atmospheric_abstract,Abstract grayscale aerial terrain depicting va...,High-contrast texture featuring granular sand ...,Generate low to moderate height fragmented roc...


In [32]:
cluster_prompts_df = pd.read_csv(API_OUTPUTS_DIR / "cluster_prompts.csv")
cluster_prompts_df

,cluster_id,cluster_name,image_prompt,td_texture_prompt,blender_fragment_prompt
0,0,dry_exposed,"abstract grayscale terrain showing rugged, por...",high contrast erosion pattern with layered gra...,dense cluster of varied height fragments with ...
1,1,frozen_quiet,vast monochrome desert terrain with gentle und...,generate sparse granular texture with low cont...,create low-height uneven terrain fragments wit...
2,2,lush_dense,abstract monochrome terrain of cracked and fra...,generate high contrast erosion patterns with s...,create dense fields of irregular flat fragment...
3,3,flowing_unstable,"abstract monochrome terrain of a deep, narrow ...",high contrast procedural erosion texture simul...,fragmented vertical rock forms with high heigh...
4,4,rocky_vertical,Abstract terrain of rugged vertical rock faces...,Generate a high contrast procedural texture of...,Create fragmented vertical rock formations wit...
5,5,atmospheric_abstract,Abstract grayscale aerial terrain depicting va...,High-contrast texture featuring granular sand ...,Generate low to moderate height fragmented roc...


In [33]:
cluster_prompts_df[["cluster_name", "td_texture_prompt", "blender_fragment_prompt"]]

,cluster_name,td_texture_prompt,blender_fragment_prompt
0,dry_exposed,high contrast erosion pattern with layered gra...,dense cluster of varied height fragments with ...
1,frozen_quiet,generate sparse granular texture with low cont...,create low-height uneven terrain fragments wit...
2,lush_dense,generate high contrast erosion patterns with s...,create dense fields of irregular flat fragment...
3,flowing_unstable,high contrast procedural erosion texture simul...,fragmented vertical rock forms with high heigh...
4,rocky_vertical,Generate a high contrast procedural texture of...,Create fragmented vertical rock formations wit...
5,atmospheric_abstract,High-contrast texture featuring granular sand ...,Generate low to moderate height fragmented roc...


In [34]:
selected_clusters = cluster_prompts_df[
    cluster_prompts_df["cluster_name"].isin([
        "dry_exposed",
        "flowing_unstable",
        "rocky_vertical"
    ])
]
selected_clusters

,cluster_id,cluster_name,image_prompt,td_texture_prompt,blender_fragment_prompt
0,0,dry_exposed,"abstract grayscale terrain showing rugged, por...",high contrast erosion pattern with layered gra...,dense cluster of varied height fragments with ...
3,3,flowing_unstable,"abstract monochrome terrain of a deep, narrow ...",high contrast procedural erosion texture simul...,fragmented vertical rock forms with high heigh...
4,4,rocky_vertical,Abstract terrain of rugged vertical rock faces...,Generate a high contrast procedural texture of...,Create fragmented vertical rock formations wit...
